# Physiology and the Regulation of Living Systems Workflow

This notebook scaffold supports the article **Physiology and the Regulation of Living Systems**. It can be expanded with balance equations, feedback control, hormonal signaling, effector response, stress scenarios, regulatory scoring, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
scenarios = pd.read_csv(article_dir / 'data' / 'feedback_scenarios.csv')
thresholds = pd.read_csv(article_dir / 'data' / 'regulatory_thresholds.csv')
scenarios.head(), thresholds

In [ ]:
def simulate_feedback(X0=10, X_star=5, I_in=0.6, a=0.9, b=0.5, c=0.7, d=0.4, u0=0.3, u1=0.25, T=40, dt=0.05):
    time = np.arange(0, T + dt, dt)
    X = np.zeros(len(time))
    H = np.zeros(len(time))
    E = np.zeros(len(time))
    X[0] = X0
    H[0] = 0
    E[0] = 0
    for t in range(1, len(time)):
        uptake = u0 + u1 * H[t - 1] * X[t - 1]
        dX = I_in - uptake
        dH = a * (X[t - 1] - X_star) - b * H[t - 1]
        dE = c * H[t - 1] - d * E[t - 1]
        X[t] = max(0, X[t - 1] + dX * dt)
        H[t] = max(0, H[t - 1] + dH * dt)
        E[t] = max(0, E[t - 1] + dE * dt)
    return pd.DataFrame({'time': time, 'regulated_variable': X, 'hormonal_signal': H, 'effector_response': E})

series = simulate_feedback()
series.head().round(3)

In [ ]:
diagnostics = {
    'peak_X': series['regulated_variable'].max(),
    'peak_H': series['hormonal_signal'].max(),
    'peak_E': series['effector_response'].max(),
    'final_X': series['regulated_variable'].iloc[-1],
    'recovery_error': abs(series['regulated_variable'].iloc[-1] - 5)
}
pd.Series(diagnostics).round(3)